# Chapter 1 — Classical Systems and Information

Programming companion to the full lecture notes by Dr. Ashutosh Bhatia. This notebook uses Python/NumPy for classical models; Qiskit is introduced when we reach quantum circuits. No quantum hardware account is needed.

Work through the corresponding theory before executing each section. Predict an answer, run the smallest example, then explain why it agrees or disagrees with your prediction. The assertions check selected mathematical properties; they do not replace the written proofs.

Sources: instructor foundations slides, PDF pages 3–13; newly authored examples and exercises aligned with Chapter 1. Uses column distributions and column-stochastic matrices throughout.


In [ ]:
%pip -q install "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("NumPy:", np.__version__)

rng = np.random.default_rng(545)

def is_distribution(p):
    p = np.asarray(p, dtype=float)
    return p.ndim == 1 and len(p) > 0 and np.all(p >= 0) and np.all(np.isfinite(p)) and np.isclose(p.sum(), 1)

def is_column_stochastic(M):
    M = np.asarray(M, dtype=float)
    return M.ndim == 2 and np.all(M >= 0) and np.all(np.isfinite(M)) and np.allclose(M.sum(axis=0), 1)


## 1.1 A state label and its representations

For the coordinate order (off, low, high), predict the column for low. Then reverse the order. The labelled setting stays the same. The binary code below is a separate fixed encoding.


In [ ]:
labels = ["off", "low", "high"]
selected = "low"
basis = np.eye(len(labels), dtype=int)
v = basis[:, labels.index(selected)]
reverse = list(reversed(labels))
v_reverse = basis[:, reverse.index(selected)]
code_map = {"off": "00", "low": "01", "high": "10"}
print("Order:", labels, "vector:", v)
print("Reversed:", reverse, "vector:", v_reverse)
print("Fixed binary code:", code_map[selected])
print("Unused code: 11")
assert np.array_equal(v, [0, 1, 0])
# Low happens to be the middle coordinate in both of these orders.
# Choose off or high to see the location of the 1 change.
for setting in labels:
    v1 = basis[:, labels.index(setting)]
    v2 = basis[:, reverse.index(setting)]
    assert np.array_equal(v2, v1[::-1])


**Try it:** choose `selected="off"`, and then use a five-label state set. Explain separately the one-hot vector dimension and the minimum fixed-length binary storage. For five labels, three bits suffice, with three unused strings.


## 1.2 Probabilities, counts and conditioning

Use p=(0.75,0.25). Predict the expected number of ones in 100 trials. One simulated batch is allowed to differ from that expectation. Every trial is an independent sample from the stated model.


In [ ]:
p = np.array([0.75, 0.25])
assert is_distribution(p)
assert not is_distribution([1.2, -0.2])
assert not is_distribution([0.2, 0.3])
N = 100
ones = int(rng.binomial(N, p[1]))
counts = np.array([N - ones, ones])
print("Model:", p)
print("Expected counts:", N * p)
print("Observed counts:", counts)
print("Observed frequencies:", counts / N)
assert counts.sum() == N
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].bar(["0", "1"], p, color="#286a9b")
axes[0].set(title="Model probabilities", ylabel="Probability", ylim=(0, 1))
axes[1].bar(["0", "1"], counts, color="#286a9b")
axes[1].set(title=f"One sampled batch (N={N})", ylabel="Count", ylim=(0, N))
for ax in axes:
    ax.set_xlabel("Outcome")
fig.tight_layout()
plt.show()


In [ ]:
die = np.array([1, 2, 1, 3, 2, 3], dtype=float) / 12
assert is_distribution(die)
even = np.array([False, True, False, True, False, True])
prob_even = die[even].sum()
conditional = np.where(even, die, 0) / prob_even
print("P(even):", prob_even)
print("Conditional distribution given even:", conditional)
assert np.isclose(prob_even, 2/3)
assert np.allclose(conditional, [0, 1/4, 0, 3/8, 0, 3/8])
assert is_distribution(conditional)
# Conditioning on probability zero is undefined in this elementary formula.


**Try it:** repeat the sampling cell without resetting the random generator; compare the batches. Then restart with the same seed to reproduce the sequence. Condition the die on faces {1,2,3} and calculate P(face 2 | face in {1,2,3}). Explain why the classical conditional entries are divided by the event probability.


## 1.3 Operations on distributions

Each input basis vector selects one output column. Predict a random flip with epsilon=1/4 acting on p=(3/4,1/4), then inspect the two route contributions to each output.


In [ ]:
I = np.eye(2)
X = np.array([[0., 1.], [1., 0.]])
R0 = np.array([[1., 1.], [0., 0.]])
R1 = np.array([[0., 0.], [1., 1.]])
epsilon = 0.25
B = (1 - epsilon) * I + epsilon * X
for M in [I, X, R0, R1, B]:
    assert is_column_stochastic(M)
    assert is_distribution(M @ p)
print("B =", B)
print("Output Bp:", B @ p)
assert np.allclose(B @ p, [5/8, 3/8])
B_inverse = np.linalg.inv(B)
print("Algebraic inverse =", B_inverse)
assert np.allclose(B_inverse @ B, I)
assert not is_column_stochastic(B_inverse)
assert np.linalg.matrix_rank(R0) == 1


**Try it:** build the asymmetric operation that keeps 0 unchanged and sends 1 to 0/1 with probabilities 1/3 and 2/3. Apply it to p=(1/4,3/4). Explain why a matrix inverse with negative entries is not an allowed stochastic reversal on arbitrary distributions.


## 1.4 Composition and operation order

A then B acts as B @ A on a column. First predict reset-then-flip and flip-then-reset, including their intermediate states.


In [ ]:
p = np.array([0.75, 0.25])
reset_then_flip = X @ R0
flip_then_reset = R0 @ X
print("After reset:", R0 @ p)
print("After reset then flip:", reset_then_flip @ p)
print("After flip:", X @ p)
print("After flip then reset:", flip_then_reset @ p)
assert np.allclose(reset_then_flip, R1)
assert np.allclose(flip_then_reset, R0)
assert not np.allclose(reset_then_flip, flip_then_reset)
assert np.allclose(reset_then_flip @ p, [0, 1])
assert np.allclose(flip_then_reset @ p, [1, 0])
assert np.allclose(X @ B, B @ X)
uniform = np.array([0.5, 0.5])
assert np.allclose(I @ uniform, X @ uniform)
assert not np.allclose(I, X)


In [ ]:
# Verify composition on several modelled operations.
# The general proof of stochastic closure is in the written lesson.
operations = [I, X, R0, R1, B]
for A in operations:
    for second in operations:
        product = second @ A
        assert is_column_stochastic(product)
        assert np.allclose(second @ (A @ p), product @ p)
print("Selected composition checks passed.")


## Chapter reflection

1. Describe an operation by its input-conditioned columns, not only its output for one chosen p.
2. Explain why stochastic closure does not imply commutation or reversibility.
3. State when a model of the second operation conditioned only on the intermediate state may be insufficient.
4. Revisit the written exercises and show your reasoning without using the code as a substitute.

This notebook contains classical examples. It establishes mathematical language that we will later compare with quantum amplitudes, unitary operations and measurement. It makes no quantum-advantage claim.
